# Section 2.5 OW Strategy Validation

This notebook validates the OW strategy implementation using the homework convention `qtilde = sigma * q / ADV`. It is runnable from either the repository root or the `notebooks/` directory.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    if start.name == "notebooks":
        start = start.parent
    for candidate in [start] + list(start.parents):
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root containing data/ and src/")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ALPHA_INPUT = PROJECT_ROOT / "outputs" / "alphas" / "strategy_alpha_input_h5m_rho010_H5m.csv"
SCALING_PATH = PROJECT_ROOT / "outputs" / "scaling" / "scaling_factors_20d.csv"
BIN_DIR = PROJECT_ROOT / "data" / "binSamples"
OUT_DIR = PROJECT_ROOT / "outputs" / "strategy" / "ow"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("CURRENT_DIR:", Path.cwd().resolve())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ALPHA_INPUT exists:", ALPHA_INPUT.exists())
print("SCALING_PATH exists:", SCALING_PATH.exists())
print("BIN_DIR exists:", BIN_DIR.exists())

CURRENT_DIR: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/notebooks
PROJECT_ROOT: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project
ALPHA_INPUT exists: True
SCALING_PATH exists: True
BIN_DIR exists: True


In [2]:
from src.scaling_factors import compute_or_load_scaling_factors
from src.strategy_config import OWStrategyConfig
from src.ow_strategy import (
    attach_scaling_factors,
    normalize_signed_volume,
    invert_normalized_trade,
    run_ow_strategy,
)
from src.strategy_metrics import (
    save_strategy_metrics,
    save_strategy_plots,
    compute_overall_strategy_summary,
)
from src.strategy_validation import validate_ow_strategy_output, save_strategy_validation_report

## 1. Scaling Factor Audit

Scaling factors are computed from `binSamples`: daily intraday volatility from `midEnd` returns and daily volume from `sum(abs(trade))`. The 20-day trailing averages exclude the current day.

In [3]:
scaling = compute_or_load_scaling_factors(
    bin_dir=BIN_DIR,
    output_path=SCALING_PATH,
    force_recompute=False,
    window_days=20,
)
print("Scaling shape:", scaling.shape)
display(scaling.head())

scaling_summary = scaling[["trailing_px_vol", "trailing_ADV", "scaling_window_days"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
display(scaling_summary)
print("Rows with full 20-day scaling window:", int(scaling["has_full_scaling_window"].sum()), "/", len(scaling))

Scaling shape: (12402, 8)


,stock,date,trailing_px_vol,trailing_ADV,px_vol_current_day,daily_volume_current_day,scaling_window_days,has_full_scaling_window
0,A,2019-01-02,NaN,NaN,0.000393,217804.0,0,False
1,A,2019-01-03,NaN,NaN,0.000531,485020.0,1,False
2,A,2019-01-04,NaN,NaN,0.000367,244093.0,2,False
3,A,2019-01-07,NaN,NaN,0.000364,251175.0,3,False
4,A,2019-01-08,NaN,NaN,0.000378,181329.0,4,False


,trailing_px_vol,trailing_ADV,scaling_window_days
count,11402.000000,1.140200e+04,12402.000000
mean,0.000295,4.099096e+05,19.153362
std,0.000093,9.221019e+05,3.294661
min,0.000105,4.372025e+04,0.000000
1%,0.000147,5.554296e+04,2.000000
5%,0.000179,6.902793e+04,12.000000
50%,0.000273,1.905020e+05,20.000000
95%,0.000462,1.103826e+06,20.000000
99%,0.000582,6.001964e+06,20.000000
max,0.000804,9.311842e+06,20.000000


Rows with full 20-day scaling window: 11402 / 12402


## 2. Strategy Run

The baseline uses linear OW normalization, placeholder `lambda=1.0`, and impact half-life `H_I=5` minutes. Rows without trailing sigma/ADV are kept but do not trade.

In [4]:
if not ALPHA_INPUT.exists():
    raise FileNotFoundError(f"Missing strategy alpha input: {ALPHA_INPUT}")

alpha_input = pd.read_csv(ALPHA_INPUT)
config = OWStrategyConfig(
    scaling_factors_path=str(SCALING_PATH.relative_to(PROJECT_ROOT)),
    use_scaling_factors=True,
    fallback_sigma=None,
    fallback_ADV=None,
    impact_model_type="linear",
    impact_lambda=1.0,
    impact_half_life_minutes=5.0,
    liquidate_at_close=True,
)

strategy_input = attach_scaling_factors(alpha_input, scaling, config)
print("Rows with scaling:", int(strategy_input["has_scaling"].sum()), "/", len(strategy_input))
trades = run_ow_strategy(strategy_input, config)
trades.to_csv(OUT_DIR / "ow_strategy_trades.csv", index=False)

cols = [
    "date", "time", "stock", "mid", "alpha", "alpha_dot", "sigma", "ADV", "has_scaling",
    "target_impact", "required_normalized_trade", "normalized_trade", "signed_volume",
    "position_after", "impact_after_trade", "net_pnl", "skipped_due_to_missing_scaling",
]
display(trades[cols].head())

Rows with scaling: 0 / 200000


,date,time,stock,mid,alpha,alpha_dot,sigma,ADV,has_scaling,target_impact,required_normalized_trade,normalized_trade,signed_volume,position_after,impact_after_trade,net_pnl,skipped_due_to_missing_scaling
0,2019-01-02,09:30:10,A,66.215,-0.000048,0.000000,NaN,NaN,False,-0.000024,-0.000024,0.0,0.0,0.0,0.0,0.0,True
1,2019-01-02,09:30:20,A,66.335,-0.000050,-0.000009,NaN,NaN,False,0.000040,0.000040,0.0,0.0,0.0,0.0,0.0,True
2,2019-01-02,09:30:30,A,66.335,-0.000052,-0.000010,NaN,NaN,False,0.000046,0.000046,0.0,0.0,0.0,0.0,0.0,True
3,2019-01-02,09:30:40,A,66.340,-0.000056,-0.000026,NaN,NaN,False,0.000157,0.000157,0.0,0.0,0.0,0.0,0.0,True
4,2019-01-02,09:31:00,A,66.270,-0.000046,0.000029,NaN,NaN,False,-0.000230,-0.000230,0.0,0.0,0.0,0.0,-0.0,True


## 3. OW Formula Validation

These checks verify `qtilde = sigma q / ADV`, impact dynamics, position dynamics, liquidation, and cost decomposition.

In [5]:
checks = validate_ow_strategy_output(trades, config)
checks_df = pd.DataFrame.from_dict(checks, orient="index")
display(checks_df)

valid = trades["has_scaling"] & trades["sigma"].gt(0) & trades["ADV"].gt(0)
if valid.any():
    qtilde_check = normalize_signed_volume(
        trades.loc[valid, "signed_volume"].to_numpy(),
        trades.loc[valid, "sigma"].to_numpy(),
        trades.loc[valid, "ADV"].to_numpy(),
        config.impact_model_type,
    )
    max_qtilde_error = np.nanmax(np.abs(qtilde_check - trades.loc[valid, "normalized_trade"].to_numpy()))
else:
    max_qtilde_error = np.nan
print("Max normalized trade formula error:", max_qtilde_error)

impact_error = (trades["impact_after_trade"] - (trades["impact_before_trade"] + config.impact_lambda * trades["normalized_trade"])).abs().max()
position_error = (trades["position_after"] - (trades["position_before"] + trades["signed_volume"])).abs().max()
print("Max impact dynamics error:", impact_error)
print("Max position dynamics error:", position_error)

,status,message
required_columns,PASS,missing=[]
no_critical_nans,PASS,"{'signed_volume': 0, 'normalized_trade': 0, 'p..."
scaling_coverage,WARN,rows_with_scaling=0.00%
no_trading_without_scaling,PASS,max_abs_signed_volume_without_scaling=0
normalized_trade_formula,PASS,max_error=0; model=linear
inverse_trade_formula,PASS,sample_max_error=0
impact_dynamics,PASS,max_error=0
position_dynamics,PASS,max_error=0
liquidation,PASS,max_final_abs_position=0
wealth_consistency,PASS,max_error=0


Max normalized trade formula error: nan
Max impact dynamics error: 0.0
Max position dynamics error: 0.0


## 4. Metrics and Plots

In [6]:
daily_metrics, summary = save_strategy_metrics(trades, OUT_DIR)
save_strategy_plots(trades, daily_metrics, OUT_DIR)
save_strategy_validation_report(checks, trades, summary, OUT_DIR)

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
display(daily_metrics.head())

,value
total_gross_pnl,0.0
total_net_pnl,0.0
total_signed_impact_cost_normalized,0.0
total_quadratic_impact_cost_normalized,0.0
mean_daily_net_pnl,0.0
std_daily_net_pnl,0.0
daily_sharpe,NaN
annualized_sharpe,NaN
total_signed_volume_turnover,0.0
total_notional_turnover,0.0


,date,gross_pnl,net_pnl,signed_impact_cost_normalized,quadratic_impact_cost_normalized,turnover_shares,turnover_notional,normalized_turnover,max_abs_position,max_abs_impact,max_participation_rate,mean_participation_rate,n_trades,n_stock_days,rows_with_scaling,rows_skipped,cumulative_daily_wealth,daily_drawdown
0,2019-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0,50,0,110382,0.0,0.0
1,2019-01-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0,40,0,89528,0.0,0.0


In [7]:
fig_paths = [
    FIG_DIR / "ow_cumulative_wealth.png",
    FIG_DIR / "ow_daily_net_pnl.png",
    FIG_DIR / "ow_gross_vs_net_pnl.png",
    FIG_DIR / "ow_participation_rate_histogram.png",
    FIG_DIR / "ow_signed_volume_histogram.png",
    FIG_DIR / "ow_normalized_trade_histogram.png",
    FIG_DIR / "ow_impact_histogram.png",
    FIG_DIR / "ow_sample_strategy_path.png",
    FIG_DIR / "ow_drawdown.png",
]
for p in fig_paths:
    print(p.relative_to(PROJECT_ROOT), "exists=", p.exists())

outputs/strategy/ow/figures/ow_cumulative_wealth.png exists= True
outputs/strategy/ow/figures/ow_daily_net_pnl.png exists= True
outputs/strategy/ow/figures/ow_gross_vs_net_pnl.png exists= True
outputs/strategy/ow/figures/ow_participation_rate_histogram.png exists= False
outputs/strategy/ow/figures/ow_signed_volume_histogram.png exists= True
outputs/strategy/ow/figures/ow_normalized_trade_histogram.png exists= True
outputs/strategy/ow/figures/ow_impact_histogram.png exists= True
outputs/strategy/ow/figures/ow_sample_strategy_path.png exists= True
outputs/strategy/ow/figures/ow_drawdown.png exists= True


## 5. Mechanical Sensitivity

This is not model calibration. It only checks that mechanics remain stable across plausible impact half-lives and placeholder lambda values.

In [8]:
sensitivity_rows = []
for half_life in [1.0, 5.0, 30.0, 60.0]:
    for lam in [0.5, 1.0, 2.0]:
        cfg = OWStrategyConfig(
            scaling_factors_path=str(SCALING_PATH.relative_to(PROJECT_ROOT)),
            use_scaling_factors=True,
            impact_model_type="linear",
            impact_lambda=lam,
            impact_half_life_minutes=half_life,
            liquidate_at_close=True,
        )
        tr = run_ow_strategy(strategy_input, cfg)
        summ = compute_overall_strategy_summary(tr)
        sensitivity_rows.append({"impact_half_life_minutes": half_life, "impact_lambda": lam, **summ})

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(OUT_DIR / "ow_sensitivity_summary.csv", index=False)
display(sensitivity[["impact_half_life_minutes", "impact_lambda", "total_net_pnl", "total_signed_volume_turnover", "total_normalized_turnover", "max_abs_impact", "rows_with_scaling"]])

,impact_half_life_minutes,impact_lambda,total_net_pnl,total_signed_volume_turnover,total_normalized_turnover,max_abs_impact,rows_with_scaling
0,1.0,0.5,0.0,0.0,0.0,0.0,0
1,1.0,1.0,0.0,0.0,0.0,0.0,0
2,1.0,2.0,0.0,0.0,0.0,0.0,0
3,5.0,0.5,0.0,0.0,0.0,0.0,0
4,5.0,1.0,0.0,0.0,0.0,0.0,0
5,5.0,2.0,0.0,0.0,0.0,0.0,0
6,30.0,0.5,0.0,0.0,0.0,0.0,0
7,30.0,1.0,0.0,0.0,0.0,0.0,0
8,30.0,2.0,0.0,0.0,0.0,0.0,0
9,60.0,0.5,0.0,0.0,0.0,0.0,0


## 6. Final Conclusion

The strategy mechanics are valid when the validation table has no critical failures. Economic interpretation remains provisional because `lambda=1.0` is a placeholder until calibrated OW parameters arrive from model fitting.

In [9]:
print("Trades:", OUT_DIR / "ow_strategy_trades.csv")
print("Daily metrics:", OUT_DIR / "ow_daily_metrics.csv")
print("Summary metrics:", OUT_DIR / "ow_summary_metrics.csv")
print("Validation report:", OUT_DIR / "ow_strategy_validation_report.txt")
print("Figures:", FIG_DIR)
print("Critical failures:", checks_df[checks_df["status"].eq("FAIL")].index.tolist())

Trades: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/strategy/ow/ow_strategy_trades.csv
Daily metrics: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/strategy/ow/ow_daily_metrics.csv
Summary metrics: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/strategy/ow/ow_summary_metrics.csv
Validation report: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/strategy/ow/ow_strategy_validation_report.txt
Figures: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/strategy/ow/figures
Critical failures: []
